In [1]:
import torch, time, gc
from transformers import AutoModelForCausalLM, logging

logging.set_verbosity_error()

L, H_kv, D, b = 36, 8, 128, 2
kv_per_token = 2 * L * H_kv * D * b
m_id = "Qwen/Qwen3-4B-Thinking-2507"

model = AutoModelForCausalLM.from_pretrained(m_id, torch_dtype=torch.float16, device_map="cuda")
res = [
    f"Theoretical KV Cache: {kv_per_token} B/token",
    f"Estimated VRAM capacity: {torch.cuda.mem_get_info()[0] // kv_per_token} tokens",
]


def run(seq):
    inp = torch.randint(1, 10000, (1, seq), device="cuda")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model.generate(inp, max_new_tokens=3, min_new_tokens=3, pad_token_id=0)
    return time.time() - t0, torch.cuda.max_memory_allocated() / 1024**2


t, m = run(1000)
res.append(f"Baseline Time: {t:.3f} s | Peak Mem: {m:.0f} MB")

seq = 1000
try:
    while True:
        seq += 1000
        run(seq)
except:
    res.append(f"Empirical Limit: ~{seq - 1000} tokens")

del model
gc.collect()
torch.cuda.empty_cache()
print("\n".join(res))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Theoretical KV Cache: 147456 B/token
Estimated VRAM capacity: 24892 tokens
Baseline Time: 0.458 s | Peak Mem: 7921 MB
Empirical Limit: ~12000 tokens


In [ ]:
import os
from vllm import LLM, SamplingParams
from vllm.config import KVTransferConfig

os.environ.update({"LMCACHE_CHUNK_SIZE": "256", "LMCACHE_LOCAL_CPU": "True", "LMCACHE_MAX_LOCAL_CPU_SIZE": "5.0"})


llm = LLM(
    model=m_id,
    gpu_memory_utilization=0.9,
    max_model_len=10000,
    kv_transfer_config=KVTransferConfig(kv_connector="LMCacheConnectorV1", kv_role="kv_both"),
)

tokenizer = llm.get_tokenizer()
sp = SamplingParams(max_tokens=3)

prompts = [
    "Explain CPU architecture " * 1000 + " in detail.",
    "Discuss GPU parallel processing " * 1000 + " in C language.",
]

token_counts = []
for i, p in enumerate(prompts):
    count = len(tokenizer.encode(p))
    token_counts.append(count)
    print(f"Prompt {i+1} Tokens: {count}")

res2 = []
for i, p in enumerate(prompts):
    t0 = time.time()
    llm.generate([p], sp)
    res2.append(f"Prompt {i+1} Time: {time.time() - t0:.3f} s")

del llm
gc.collect()
torch.cuda.empty_cache()
if torch.distributed.is_initialized():
    torch.distributed.destroy_process_group()

print("\n".join(res2))

INFO 05-29 11:15:32 [utils.py:240] non-default args: {'max_model_len': 10000, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'kv_transfer_config': KVTransferConfig(kv_connector='LMCacheConnectorV1', engine_id='9cdd2a7b-20ad-4305-b029-29b4e56a93a8', kv_buffer_device='cuda', kv_buffer_size=1000000000.0, kv_role='kv_both', kv_rank=None, kv_parallel_size=1, kv_ip='127.0.0.1', kv_port=14579, kv_connector_extra_config={}, kv_connector_module_path=None, enable_permute_local_kv=False, kv_load_failure_policy='fail'), 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-29 11:15:33 [model.py:568] Resolved architecture: Qwen3ForCausalLM
INFO 05-29 11:15:33 [model.py:1697] Using max model len 10000
INFO 05-29 11:15:33 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-29 11:15:33 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 05-29 11:15:33 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native']

(EngineCore pid=20249) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=20249) INFO 05-29 11:15:42 [weight_utils.py:938] Filesystem type for checkpoints: EXT4. Checkpoint size: 7.49 GiB. Available RAM: 19.02 GiB.
(EngineCore pid=20249) INFO 05-29 11:15:42 [weight_utils.py:961] Auto-prefetch is disabled because the filesystem (EXT4) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:00,  3.01it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:00<00:00,  2.88it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:00<00:00,  4.13it/s]
(EngineCore pid=20249) 


(EngineCore pid=20249) INFO 05-29 11:15:42 [default_loader.py:397] Loading weights took 0.73 seconds
(EngineCore pid=20249) INFO 05-29 11:15:43 [gpu_model_runner.py:4959] Model loading took 7.64 GiB memory and 1.731177 seconds
(EngineCore pid=20249) INFO 05-29 11:15:44 [backends.py:1089] Using cache directory: /home/student/.cache/vllm/torch_compile_cache/4a296c1127/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=20249) INFO 05-29 11:15:44 [backends.py:1148] Dynamo bytecode transform time: 1.26 s
(EngineCore pid=20249) INFO 05-29 11:15:45 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.575 s
(EngineCore pid=20249) INFO 05-29 11:15:45 [decorators.py:311] Directly load AOT compilation from path /home/student/.cache/vllm/torch_compile_cache/torch_aot_compile/124988f466779ed6f352ec68197b06010cc7fa6358d7dd50f8bb8b5b341c3945/rank_0_0/model
(EngineCore pid=20249) INFO 05-29 11:15:45 [monitor.py:53] torch.compile took 2.15 s i

(EngineCore pid=20249) [2026-05-29 11:15:47,586] LMCache INFO: Using backend: lmcache.c_ops (__init__.py:97:lmcache)
(EngineCore pid=20249) [2026-05-29 11:15:47,702] LMCache INFO: Discovered API module: env_api (router_discovery.py:53:lmcache.v1.utils.router_discovery)
(EngineCore pid=20249) [2026-05-29 11:15:47,712] LMCache INFO: Discovered API module: loglevel_api (router_discovery.py:53:lmcache.v1.utils.router_discovery)
(EngineCore pid=20249) [2026-05-29 11:15:47,712] LMCache INFO: Discovered API module: metrics_api (router_discovery.py:53:lmcache.v1.utils.router_discovery)
(EngineCore pid=20249) [2026-05-29 11:15:47,713] LMCache INFO: Discovered API module: periodic_thread_api (router_discovery.py:53:lmcache.v1.utils.router_discovery)
(EngineCore pid=20249) [2026-05-29 11:15:47,714] LMCache INFO: Discovered API module: run_script_api (router_discovery.py:53:lmcache.v1.utils.router_discovery)
(EngineCore pid=20249) [2026-05-29 11:15:47,714] LMCache INFO: Discovered API module: thre

... (10 outputs left out)

(EngineCore pid=20249) [2026-05-29 11:15:56,384] LMCache INFO: Reqid: 1-adbdc437, Total tokens 4005, Inference Engine computed tokens: 0, LMCache hit tokens: 0, need to load: 0 (vllm_v1_adapter.py:1426:lmcache.integration.vllm.vllm_v1_adapter)


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=20249) [2026-05-29 11:15:56,935] LMCache INFO: [req_id=1-adbdc437] Stored 3840 out of total 3840 tokens. size: 0.5273 GB, cost 33.0904 ms, throughput: 15.9365 GB/s; offload_time: 33.0334 ms, put_time: 0.0340 ms (cache_engine.py:553:lmcache.v1.cache_engine)


(EngineCore pid=20249) INFO 05-29 11:15:56 [core.py:1242] Shutdown initiated (timeout=0)
(EngineCore pid=20249) INFO 05-29 11:15:56 [core.py:1265] Shutdown complete
Prompt 1 Time: 0.575 s
Prompt 2 Time: 0.608 s
